# Qwen2.5-7B-Instruct → Q4_K_M GGUF
This notebook downloads Qwen2.5-7B-Instruct from HuggingFace, converts it to GGUF (F16), then quantizes it to Q4_K_M using llama.cpp.

## Step 1: Install system dependencies and clone llama.cpp

In [ ]:
!apt-get install -y build-essential cmake git
!pip install huggingface_hub
!git clone --depth=1 https://github.com/ggerganov/llama.cpp /content/llama.cpp

## Step 2: Install Python dependencies

In [ ]:
!pip install torch transformers sentencepiece protobuf numpy gguf

## Step 3: Download Qwen2.5-7B-Instruct from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download
import os

# If the model is gated, uncomment and set your HF token:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

model_id = "Qwen/Qwen2.5-7B-Instruct"
local_dir = "/content/qwen2.5-7b-instruct"

snapshot_download(
    repo_id=model_id,
    local_dir=local_dir,
    ignore_patterns=["*.bin", "*.pt"]  # download safetensors only
)
print("Download complete!")

## Step 4: Convert HuggingFace model to GGUF (F16)

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/qwen2.5-7b-instruct \
    --outfile /content/qwen2.5-7b-instruct-f16.gguf \
    --outtype f16

## Step 5: Download pre-built llama.cpp binaries (latest release)

In [ ]:
import requests

# Get the latest release tag from GitHub
release = requests.get("https://api.github.com/repos/ggerganov/llama.cpp/releases/latest").json()
tag = release["tag_name"]
print(f"Latest llama.cpp release: {tag}")

# Download the Ubuntu x64 tarball
url = f"https://github.com/ggerganov/llama.cpp/releases/download/{tag}/llama-{tag}-bin-ubuntu-x64.tar.gz"
print(f"Downloading: {url}")
!wget -q "{url}" -O /content/llama.tar.gz

# Extract
!mkdir -p /content/llama-bin
!tar -xzf /content/llama.tar.gz -C /content/llama-bin

# Find the quantize binary
!find /content/llama-bin -name "llama-quantize"

## Step 6: Quantize to Q4_K_M

> **Note:** The binary path below includes the release tag (e.g. `llama-b8234`). If the tag printed in Step 5 is different, update the path accordingly.

In [ ]:
import glob

# Auto-detect the quantize binary path
matches = glob.glob("/content/llama-bin/**/llama-quantize", recursive=True)
if not matches:
    raise FileNotFoundError("llama-quantize not found! Check the extraction in Step 5.")
quantize_bin = matches[0]
print(f"Using: {quantize_bin}")

!chmod +x "{quantize_bin}"
!"{quantize_bin}" \
    /content/qwen2.5-7b-instruct-f16.gguf \
    /content/qwen2.5-7b-instruct-Q4_K_M.gguf \
    Q4_K_M

## Step 7: Save to Google Drive

Mount your Drive and copy the quantized file there. Recommended over direct download for a ~4.4 GB file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/qwen2.5-7b-instruct-Q4_K_M.gguf "/content/drive/MyDrive/qwen2.5-7b-instruct-Q4_K_M.gguf"
print("Done! File saved to Google Drive.")

## (Optional) Step 8: Direct download to your computer

Only use this if you don't have Drive access. Note: downloading ~4.4 GB directly from Colab can be slow and may time out.

In [ ]:
from google.colab import files
files.download("/content/qwen2.5-7b-instruct-Q4_K_M.gguf")